# Step 1: Chunk data

In [2]:
from utils import load_json, chunk_clause

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
law_data = load_json('data/alqac25_law.json')

In [3]:
clause_chunk = chunk_clause(data=law_data, output_file=f'data/private/alqac25_law_clause.json')

Chunked 28 laws into 14475 chunks.
✅ Chunked data saved to data/private/alqac25_law_clause.json


# Step 2: Embedding data

In [4]:
from utils import data_to_dict, embed_data, create_hybrid_collection, insert_data_with_vectors

In [5]:
data_for_embedding = data_to_dict(clause_chunk)

Converted data to dictionary with 14475 entries.


In [6]:
embedding_data = embed_data(data=data_for_embedding, output_file='data/private/embeddings_clause.json')

CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 3050 Laptop GPU
Using device: cuda:0
Model device: cuda:0


Processing batches: 100%|██████████| 905/905 [15:43<00:00,  1.04s/it]  


✅ Embeddings saved to data/private/embeddings_clause.json


In [1]:
name = f"alqac_law_collection_clause"

In [8]:
create_hybrid_collection(name)

c:\Python310\lib\site-packages\weaviate\collections\classes\config.py:1950: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for cls_field in self.model_fields:


✅ Created schema 'alqac_law_collection_clause' successfully.


In [9]:
insert_data_with_vectors(name, embedding_data)

Total vectors in file: 14475
Successfully inserted: 14475
Failed to insert: 0
Total processed: 14475


# Step 3: Rewrite Query

In [3]:
from sentence_transformers.cross_encoder import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import get_json, apply_rules_with_overlap, normalize, similarity

In [ ]:
path = "./data/alqac25_private_test_task1.json"
data = load_json(path)

In [ ]:
model_name = 'AITeamVN/GRPO-VI-Qwen2-7B-RAG'

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    use_cache=True
)


In [ ]:
prompt_template = """
Bạn là một chuyên gia pháp lý, có khả năng đọc hiểu và phân tích các câu hỏi liên quan đến pháp luật Việt Nam.

### Hướng dẫn:
1. Từ đoạn văn bản hoặc câu hỏi ban đầu của người dùng, hãy xác định phần thông tin nào là **khách quan, trung lập, không chứa quan điểm hoặc cảm xúc cá nhân** — đây là phần dùng làm **ngữ cảnh pháp lý** để truy xuất văn bản.
2. Đồng thời, xác định rõ **câu hỏi cụ thể** mà người dùng muốn được giải đáp — loại bỏ các yếu tố thiên kiến hoặc cảm xúc trong câu hỏi nếu có.
3. Viết lại phần câu hỏi dưới dạng **câu truy vấn pháp lý**, dạng khẳng định, dùng để tìm kiếm văn bản pháp luật liên quan.
4. Truy vấn cần sử dụng đúng các cụm pháp lý như: “quy định pháp luật về…”, “trách nhiệm của…”, “điều kiện để…”, v.v.
5. Kết quả đầu ra phải ở định dạng JSON gồm hai khóa: `"question"` (câu hỏi người dùng) và `"query"` (truy vấn pháp lý đã viết lại).

### Ví dụ:

#### Ví dụ 1:
{{
  "question": "Nguyên tắc bảo đảm quyền tiếp cận thông tin theo quy định của Luật Tiếp cận thông tin bao gồm mấy nguyên tắc?",
  "query": "Các nguyên tắc bảo đảm quyền tiếp cận thông tin theo quy định của Luật Tiếp cận thông tin."
}}

#### Ví dụ 2:
{{
  "question": "Trường hợp công dân không trong độ tuổi nhập ngũ, nếu đi du học, xuất khẩu lao động không cần phải khai báo tạm vắng",
  "query": "Quy định pháp luật về việc công dân không trong độ tuổi nhập ngũ đi du học hoặc xuất khẩu lao động có phải khai báo tạm vắng hay không."
}}

#### Ví dụ 3:
{{
  "question": "Nguyên tắc đối thoại trong vụ án hành chính được quy định như thế nào cho đúng với quy định của pháp luật?",
  "query": "Quy định pháp luật về nguyên tắc đối thoại trong vụ án hành chính."
}}

#### Ví dụ 4:
{{
  "question": "Người giám hộ KHÔNG có nghĩa vụ nào sau đây đối với người được giám hộ từ đủ mười lăm tuổi đến chưa đủ mười tám tuổi?",
  "query": "Nghĩa vụ của người giám hộ đối với người được giám hộ từ đủ mười lăm tuổi đến chưa đủ mười tám tuổi."
}}

### Đầu vào:
{text}

### Kết quả:
"""

In [ ]:
eos = tokenizer.eos_token if tokenizer.eos_token else tokenizer.pad_token

In [ ]:
for item in tqdm(data , desc="Rewrite Query"):
    prompt = prompt_template.format(text=item["text"])
    inputs = tokenizer([prompt + eos], return_tensors="pt").to("cuda")

    max_retry = 10
    retry_count = 0

    while retry_count < max_retry:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=196,
                use_cache=True,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
            response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        query = get_json(response[0])
        if query and normalize(query) != normalize(item["text"]):
            break 

        retry_count += 1
    query1 = apply_rules_with_overlap(query, rule_map)
    q = item["text"]
    item["text"] = query1
    print("Count:", retry_count + 1)
    print("Response:", response[0])
    print("Question:", q)
    print("Output:", query)
    print("Query:", query1)

In [ ]:
with open("/data/private/new_query1.json", "w", encoding="utf8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Step 4: Retrieve 

In [ ]:
from utils import retrieve

In [ ]:
name = "alqac_law_collection_clause"

In [ ]:
## File 3 Output

retrieve(input_file="data/private/new_query1.json",
        output_file="data/private/chunk500_norerank_rewritequery_top1_PRIVATE.json",
        name=name,
        mode="hybrid",
        top_k=1,
        alpha=0.8)

In [ ]:
name = "alqac_law_collection_chunk_500"

In [ ]:
retrieve(input_file="data/private/new_query1.json",
        output_file="data/private/retrieval_candidates_rechunked_500_newquery_PRIVATE.json",
        name=name,
        mode="hybrid",
        top_k=12,
        alpha=0.8)

# Step 4: Rerank

In [ ]:
from utils import rerank

In [11]:
## Output File 2

rerank(candidate_file="data/private/retrieval_candidates_rechunked_500_newquery_PRIVATE.json",
       output_file=f"data/private/final_reranked_chunk__with_split_newquery_results_PRIVATE.json",
       threshold_mode="dynamic",
       threshold_value=1,
       batch_size=8)

NameError: name 'rerank' is not defined